In [ ]:
#|default_exp augment

# Load deps

In [ ]:
# !pip install -q torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')

import sys
app_path = '/content/drive/MyDrive/Projects/miniSD'
sys.path.append(app_path)

In [ ]:
#|export
import torch,random
from torch import nn
from torch.nn import init

In [ ]:
import matplotlib as mpl
from pathlib import Path
from functools import partial
import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,optim
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset
from torch import distributions
from torchvision import transforms

from src.utils import set_seed
from src.datasets import inplace, DataLoaders, show_images
from src.learner import (
    DeviceCB, MetricsCB, ProgressCB, TrainLearner,
    SingleBatchCB, Learner, Callback, to_cpu
)
from src.activations import ActivationStats, Hooks
from src.init import GeneralRelu, init_weights, conv
from src.sgd import BatchSchedCB
from src.init import clean_mem, BatchTransformCB
from src.resnet import ResBlock
from src.utils import in_notebook, noop

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
set_seed(42)
ds_name = "fashion_mnist"
ds_xl,ds_yl = 'image','label'
bs = 256
num_dl_workers = 2
lr,epochs = 6e-2,5

# Load data

In [ ]:
dsd = load_dataset(ds_name)

train_input_avg = dsd["train"]\
  .map(
      lambda x: {
          "input_mean": [
              TF.to_tensor(image).mean() for image in x[ds_xl]
          ],
          "input_std": [
              TF.to_tensor(image).std() for image in x[ds_xl]
          ]
      },
      batched=True,
      batch_size=10000,
      remove_columns=dsd["train"].column_names,
  )

input_mean_df = train_input_avg.to_pandas().mean()
xmean = input_mean_df["input_mean"]
xstd = input_mean_df["input_std"]


@inplace
def transformi(b): b[ds_xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[ds_xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)

# Set up callbacks

In [ ]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
# astats = ActivationStats(lambda mod: isinstance(mod, GeneralRelu))
cbs = [
    DeviceCB(), metrics, ProgressCB(plot=True),
    # astats
]
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)

# Improve model architecture

## Going wider

In [ ]:
def get_model(act=nn.ReLU, nfs=(16,32,64,128,256,512), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [nn.Flatten(), nn.Linear(nfs[-1], 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)


In [ ]:
# learn.fit(epochs)
clean_mem()

## Pooling

In [ ]:
class GlobalAvgPool(nn.Module):
    def forward(self, x): return x.mean((-2,-1))

def get_model2(act=nn.ReLU, nfs=(16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    # replaced the last ResNet with stride=2 with a ResNet with [stride=1 + Pooling Block]
    layers += [ResBlock(256, 512, act=act, norm=norm), GlobalAvgPool()]
    layers += [nn.Linear(512, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
#|export --> learner
def _flops(x, h, w):
    if x.dim()<3: return x.numel()
    if x.dim()==4: return x.numel()*h*w

def summary(self:Learner):
    res = '|Module|Input|Output|Num params|MFLOPS|\n|--|--|--|--|--|\n'
    totp,totf = 0,0
    def _f(hook, mod, inp, outp):
        nonlocal res,totp,totf
        nparms = sum(o.numel() for o in mod.parameters())
        totp += nparms
        *_,h,w = outp.shape
        flops = sum(_flops(o, h, w) for o in mod.parameters())/1e6
        totf += flops
        res += f'|{type(mod).__name__}|{tuple(inp[0].shape)}|{tuple(outp.shape)}|{nparms}|{flops:.1f}|\n'
    with Hooks(self.model, _f) as hooks: self.fit(1, lr=1, train=False, cbs=[SingleBatchCB()])
    print(f"Tot params: {totp}; MFLOPS: {totf:.1f}")
    if in_notebook():
        from IPython.display import Markdown
        return Markdown(res)
    else: print(res)

Learner.summary = summary

In [ ]:
TrainLearner(get_model2(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

In [ ]:
set_seed(42)
model = get_model2(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [ ]:
# learn.fit(epochs)
clean_mem()

## Simplify
- less number of params --> less memory usage
- les MFLOPS --> less compute

Note: these two numbers can change almost independently

In [ ]:
def get_model3(act=nn.ReLU, nfs=(16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    # removed tha last resnet block
    layers += [GlobalAvgPool(), nn.Linear(256, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
TrainLearner(get_model3(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

In [ ]:
first_mod_parmas = [o.shape for o in get_model3()[0].parameters()]
print(first_mod_parmas)
print(sum([x.numel() for x in first_mod_parmas]))

In [ ]:
set_seed(42)
model = get_model3(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [ ]:
# learn.fit(epochs)
clean_mem()

## Reduce MFLOPS further

In [ ]:
def get_model4(act=nn.ReLU, nfs=(16,32,64,128,256), norm=nn.BatchNorm2d):
    # replaced the first ResNet block with a simple conv block
    layers = [conv(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [GlobalAvgPool(), nn.Linear(256, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [ ]:
TrainLearner(get_model4(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

In [ ]:
first_mod_parmas = [o.shape for o in get_model4()[0].parameters()]
print(first_mod_parmas)
print(sum([x.numel() for x in first_mod_parmas]))

In [ ]:
set_seed(42)
model = get_model4(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [ ]:
# learn.fit(epochs)
clean_mem()

# Data augmentation

- After 20 epochs without augmentation:

```
{'accuracy': '0.999', 'loss': '0.012', 'epoch': 19, 'train': True}
{'accuracy': '0.924', 'loss': '0.284', 'epoch': 19, 'train': False}
```
- We need to **regularize**
    - With **batchnorm**, weight decay **doesn't** really regularize. #TODO: WHY?

## Train time augmentation

In [ ]:
def tfm_batch(b, tfm_x=noop, tfm_y = noop): return tfm_x(b[0]),tfm_y(b[1])

tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=4),
    transforms.RandomHorizontalFlip()
) # these are happening on GPU

augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[SingleBatchCB(), augcb])
learn.fit(1)

In [ ]:
xb,yb = learn.batch
show_images(xb[:16], imsize=1.5);

In [ ]:
#| export --> leaner
def show_image_batch(self:Learner, max_n=9, cbs=[], **kwargs):
    self.fit(1, cbs=[SingleBatchCB()]+cbs)
    show_images(self.batch[0][:max_n], **kwargs)

Learner.show_image_batch = show_image_batch

In [ ]:
learn.show_image_batch(max_n=16, imsize=(1.5))

In [ ]:
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip()
)
# TODO: A custom collation function could let you do per-item transformations.
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)
set_seed(42)
aug_epochs = 10 # keep runs fast and light
lr = 1e-2
tmax = aug_epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)

In [ ]:
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
# learn.fit(aug_epochs)

In [ ]:
# mdl_path = Path('models')
# mdl_path.mkdir(exist_ok=True)
# torch.save(learn.model, mdl_path/'data_aug.pkl')

In [ ]:
mdl_path = Path('models')
model = torch.load(mdl_path/'data_aug.pkl',weights_only=False)
learn = TrainLearner(
    model, dls, F.cross_entropy, lr=lr,
    cbs=cbs+xtra,
    opt_func=optim.AdamW
)

## Test time augmentation (TTA)

In [ ]:
#| export
class CapturePreds(Callback):
    def before_fit(self, learn): self.all_inps,self.all_preds,self.all_targs = [],[],[]
    def after_batch(self, learn):
        self.all_inps.append(to_cpu(learn.batch[0]))
        self.all_preds.append(to_cpu(learn.preds))
        self.all_targs.append(to_cpu(learn.batch[1]))
    def after_fit(self, learn):
        self.all_preds,self.all_targs,self.all_inps = map(torch.cat, [self.all_preds,self.all_targs,self.all_inps])

def capture_preds(self: Learner, cbs=[], inps=False):
    cp = CapturePreds()
    self.fit(1, train=False, cbs=[cp]+cbs)
    res = cp.all_preds,cp.all_targs
    if inps: res = res+(cp.all_inps,)
    return res
Learner.capture_preds = capture_preds

In [ ]:
pcb = list(filter(lambda cb: isinstance(cb, ProgressCB), learn.cbs))[0]
pcb.plot = False

In [ ]:
ap1, at = learn.capture_preds()
ttacb = BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)
ap2, at = learn.capture_preds(cbs=[ttacb])
print(ap1.shape,ap2.shape,at.shape)
ap = torch.stack([ap1,ap2]).mean(0).argmax(1)
ensemble_acc = round((ap==at).float().mean().item(), 3)
print(f"ensemble_acc: {ensemble_acc}")

## Train time augmentation (continued)

In [ ]:
pcb.plot = True

### Random erase

In [ ]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
print(f"mean: {xbt.mean()} | std: {xbt.std()} | min: {xbt.min()} | max: {xbt.max()}")
pct = 0.2
# size of the erases box in each dimension
szx = int(pct*xbt.shape[-2])
szy = int(pct*xbt.shape[-1])
# starting point of the erases box in each dimension
stx = int(random.random()*(1-pct)*xbt.shape[-2])
sty = int(random.random()*(1-pct)*xbt.shape[-1])
init.normal_(xbt[:,:,stx:stx+szx,sty:sty+szy], mean=xbt.mean(), std=xbt.std());
show_images(xbt, imsize=1.5)
print(f"mean: {xbt.mean()} | std: {xbt.std()} | min: {xbt.min()} | max: {xbt.max()}")

In [ ]:
#|export
def _rand_erase1(x, pct, xm, xs, mn, mx):
    szx = int(pct*x.shape[-2])
    szy = int(pct*x.shape[-1])
    stx = int(random.random()*(1-pct)*x.shape[-2])
    sty = int(random.random()*(1-pct)*x.shape[-1])
    init.normal_(x[:,:,stx:stx+szx,sty:sty+szy], mean=xm, std=xs)
    # this is the fix: forcing the same range on new randomly generated pixels
    x.clamp_(mn, mx)

In [ ]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
print(f"mean: {xbt.mean()} | std: {xbt.std()} | min: {xbt.min()} | max: {xbt.max()}")
_rand_erase1(xbt, 0.2, xbt.mean(), xbt.std(), xbt.min(), xbt.max())
show_images(xbt, imsize=1.5)
print(f"mean: {xbt.mean()} | std: {xbt.std()} | min: {xbt.min()} | max: {xbt.max()}")

In [ ]:
#|export
def rand_erase(x, pct=0.2, max_num = 4):
    xm,xs,mn,mx = x.mean(),x.std(),x.min(),x.max()
    num = random.randint(0, max_num)
    # print(f"inside rand_erase: num = {num}")
    for i in range(num): _rand_erase1(x, pct, xm, xs, mn, mx)
    return x

In [ ]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
show_images(rand_erase(xbt, 0.2, 4), imsize=1.5);

In [ ]:
#|export
class RandErase(nn.Module):
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct,self.max_num = pct,max_num
    def forward(self, x): return rand_erase(x, self.pct, self.max_num)

In [ ]:
#|export --> learner
class CompletionCB(Callback):
    def before_fit(self, learn): self.count = 0
    def after_batch(self, learn): self.count += 1
    def after_fit(self, learn): print(f'Completed {self.count} batches')

In [ ]:
tfms = nn.Sequential(
    transforms.RandomCrop(28, padding=1),
    transforms.RandomHorizontalFlip(),
    RandErase()
)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

model = get_model()
learn = TrainLearner(
    model, dls, F.cross_entropy, lr=lr,
    cbs=[DeviceCB(), SingleBatchCB(), augcb, CompletionCB()]
)
learn.fit(1, valid=False)
xb,yb = learn.batch
show_images(xb[:16], imsize=1.5);

In [ ]:
lr = 2e-2
tmax = aug_epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(aug_epochs)

### Random copy

- In order to end up with the **exact same distribution** of the initial image,
    - we can copy another piece of the same image instead of erasing it.
- No need to adjust the min and max (clamping) afterward

In [ ]:
#|export
def _rand_copy1(x, pct):
    szx = int(pct*x.shape[-2])
    szy = int(pct*x.shape[-1])
    stx1 = int(random.random()*(1-pct)*x.shape[-2])
    sty1 = int(random.random()*(1-pct)*x.shape[-1])
    stx2 = int(random.random()*(1-pct)*x.shape[-2])
    sty2 = int(random.random()*(1-pct)*x.shape[-1])
    x[:,:,stx1:stx1+szx,sty1:sty1+szy] = x[:,:,stx2:stx2+szx,sty2:sty2+szy]

In [ ]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
print(f"mean: {xbt.mean()} | std: {xbt.std()} | min: {xbt.min()} | max: {xbt.max()}")
_rand_copy1(xbt, 0.2)
print(f"mean: {xbt.mean()} | std: {xbt.std()} | min: {xbt.min()} | max: {xbt.max()}")
show_images(xbt, imsize=1.5);

In [ ]:
#|export
def rand_copy(x, pct=0.2, max_num = 4):
    num = random.randint(0, max_num)
    for i in range(num): _rand_copy1(x, pct)
    return x

class RandCopy(nn.Module):
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct,self.max_num = pct,max_num
    def forward(self, x): return rand_copy(x, self.pct, self.max_num)

In [ ]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
rand_copy(xbt, 0.2, 4)
show_images(xbt, imsize=1.5);

In [ ]:
tfms = nn.Sequential(transforms.RandomCrop(28, padding=1),
                     transforms.RandomHorizontalFlip(),
                     RandCopy())
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[DeviceCB(), SingleBatchCB(), augcb, CompletionCB()])
learn.fit(1, valid=False)
xb,yb = learn.batch
show_images(xb[:16], imsize=1.5);

In [ ]:
set_seed(1)
lr = 1e-2
tmax = aug_epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(aug_epochs)

In [ ]:
model2 = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn2 = TrainLearner(model2, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn2.fit(aug_epochs)

#### Ensemble

In [ ]:
# mdl_path = Path('models')
# torch.save(learn.model,  mdl_path/'randcopy1.pkl')
# torch.save(learn2.model, mdl_path/'randcopy2.pkl')

In [ ]:
pcb.plot = False

In [ ]:
cp1 = [CapturePreds()]
learn.fit(1, train=False, cbs=cp1)

cp2 = [CapturePreds()]
learn2.fit(1, train=False, cbs=cp2)

ap = torch.stack([cp1[0].all_preds,cp2[0].all_preds]).mean(0).argmax(1)
round((ap==cp1[0].all_targs).float().mean().item(), 3)